In [ ]:
from villas.dataprocessing.readtools import *
from villas.dataprocessing.timeseries import *
from villas.dataprocessing.timeseries import TimeSeries as ts
import matplotlib.pyplot as plt
import re
import os
import urllib.request

# %matplotlib widget

# EMT, DP and SP simulation of a topology with slack, line and VSI, with a frequency ramp

## DP and SP simulation

In [ ]:
%%bash
TOP=${TOP:-$(git rev-parse --show-toplevel)}
PATH=${TOP}/build/dpsim/examples/cxx

DURATION=8.0
TIMESTEP=3e-3

DP_Slack_PiLine_VSI_Ramp_with_PF_Init --timestep=${TIMESTEP} --duration=${DURATION}
SP_Slack_PiLine_VSI_Ramp_with_PF_Init --timestep=${TIMESTEP} --duration=${DURATION}
EMT_Slack_PiLine_VSI_Ramp_with_PF_Init --timestep=50e-6 --duration=${DURATION}

In [ ]:
modelName = "DP_Slack_PiLine_VSI_Ramp_with_PF_Init"
path = "logs/" + modelName + "/"
dpsim_result_file = path + modelName + ".csv"

ts_dpsim = read_timeseries_csv(dpsim_result_file)
ts_dpsim_shifted = ts.frequency_shift_list(ts_dpsim, 50)

sp_model_name = "SP_Slack_PiLine_VSI_Ramp_with_PF_Init"
ts_dpsim_sp = read_timeseries_csv(
    "logs/" + sp_model_name + "/" + sp_model_name + ".csv"
)

## EMT reference

In [ ]:
emt_ref_file = (
    "logs/EMT_Slack_PiLine_VSI_Ramp_with_PF_Init/"
    "EMT_Slack_PiLine_VSI_Ramp_with_PF_Init.csv"
)
ts_dpsim_ref_emt = read_timeseries_csv(emt_ref_file)
emt_reference_available = True

## Plots

In [ ]:
plt.figure(figsize=(12, 6))
for ts_name, ts_obj in ts_dpsim.items():
    if ts_name in ["f_src"]:
        plt.plot(ts_obj.time, ts_obj.values, label=ts_name, color="C0")

plt.xlim(4.8, 8)
plt.xlabel("Zeit [s]")
plt.ylabel("Frequenz der Spannungsquelle [Hz]")
plt.gcf().legend(loc="upper center", ncol=2)
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
if emt_reference_available:
    ts_obj = ts_dpsim_ref_emt["v2_0"]
    plt.plot(ts_obj.time, np.sqrt(3 / 2) * ts_obj.values, label="u2 (EMT, 50 µs)")
ts_obj = ts_dpsim["v2"]
plt.plot(
    ts_obj.time,
    ts_obj.abs().values,
    label="u2 abs (DP, 3 ms)",
    color="C1",
    linestyle="--",
)

plt.xlim(4.8, 8)
plt.ylim(20014, 20030)
plt.xlabel("Time [s]")
plt.ylabel("Load voltage magnitude v2 [V]")
plt.gcf().legend(loc="upper center", ncol=3)
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
if emt_reference_available:
    ts_obj = ts_dpsim_ref_emt["v2_0"]
    plt.plot(ts_obj.time, np.sqrt(3 / 2) * ts_obj.values, label="u2 (EMT, 50 µs ref)")

ts_v2_interpolated = ts_dpsim["v2"].interpolate(50e-6)
ts_v2_shifted = ts.frequency_shift(ts_v2_interpolated, 50)
plt.plot(
    ts_v2_shifted.time,
    ts_v2_shifted.values,
    label="u2 (DP, 50 µs interpolated, shifted)",
    color="C1",
)
ts_v2_shifted_sp_plot = ts.frequency_shift(ts_dpsim_sp["v2"].interpolate(50e-6), 50)
plt.plot(
    ts_v2_shifted_sp_plot.time,
    ts_v2_shifted_sp_plot.values,
    label="u2 (SP, 50 µs interpolated, shifted)",
    color="C2",
)

plt.xlim(4.8, 8)
# plt.xlim(6.0,6.1)
plt.ylim(20014, 20030)
plt.xlabel("Time [s]")
plt.ylabel("Load voltage magnitude [V]")
plt.gcf().legend(loc="upper center", ncol=3)
plt.grid("y")
plt.show()

### Assertion

In [ ]:
window = slice(80000, 160002)
reference = np.sqrt(3 / 2) * ts_dpsim_ref_emt["v2_0"].values[window]

ts_v2_shifted_sp = ts.frequency_shift(ts_dpsim_sp["v2"].interpolate(50e-6), 50)

error_abs_dp = np.absolute(reference - ts_v2_shifted.values[window]).max()
print("EMT v2_0 vs. DP v2_shift (abs): " + str(error_abs_dp))
assert error_abs_dp < 10

error_abs_sp = np.absolute(reference - ts_v2_shifted_sp.values[window]).max()
print("EMT v2_0 vs. SP v2_shift (abs): " + str(error_abs_sp))
assert error_abs_sp < 10